In [1]:
# ── CELL 1 FIX: Tất cả regex và helper đã sửa ────────────────────────────────
import re, json
from typing import Any
from urllib.parse import urljoin, urlparse

# ── Regex contact ─────────────────────────────────────────────────────────────
_PHONE_RE     = re.compile(r"tel:([\d\s\-\+\.]+)", re.IGNORECASE)
_ZALO_RE      = re.compile(r"zalo\.me/([^\s\)\"]+)", re.IGNORECASE)
_EMAIL_RE     = re.compile(r"mailto:([^\s\)\"]+)", re.IGNORECASE)
_FB_RE        = re.compile(r"facebook\.com/([^\s\)\"]+)", re.IGNORECASE)
_TIKTOK_RE    = re.compile(r"tiktok\.com/@([^\s\)\"]+)", re.IGNORECASE)
_INSTAGRAM_RE = re.compile(r"instagram\.com/([^\s\)\"]+)", re.IGNORECASE)
_ADDRESS_RE   = re.compile(
    r"(số\s+\d+.{5,80}(?:đường|phố|quận|huyện|tp\.|thành phố)[^,\n]{0,60})",
    re.IGNORECASE,
)
_HOURS_RE = re.compile(r"(\d{1,2}[h:]\d{0,2}\s*[-–]\s*\d{1,2}[h:]\d{0,2})", re.IGNORECASE)

# ── Regex cấu trúc markdown ───────────────────────────────────────────────────
_H1_RE = re.compile(r"^#\s+(.+)$", re.MULTILINE)
_H2_RE = re.compile(r"^##\s+(.+)$", re.MULTILINE)

# FIX 4: Bắt [label](url) và [label](url "title") — chỉ lấy url, bỏ title attribute
_MD_LINK_RE = re.compile(
    r'\[([^\]]+)\]\(((?:/[^\s\)\"]*|https?://[^\s\)\"]*))(?:\s+"[^"]*")?\)',
    re.IGNORECASE,
)

# FIX 2: Tagline không được chứa markdown link syntax
_TAGLINE_RE = re.compile(
    r'(?<!\])\((?!https?://)'  # không bắt đầu bằng markdown link
    r'|((?:chuyên|cung cấp|được thành lập|với hơn|hệ thống|chuỗi)[^)\n\]]{20,200})',
    re.IGNORECASE,
)
# Dùng pattern đơn giản hơn, an toàn hơn:
_TAGLINE_SAFE_RE = re.compile(
    r'^(?!.*\]\()'  # loại dòng có markdown link
    r'.*(chuyên|cung cấp|được thành lập|với hơn|hệ thống|chuỗi).{20,200}$',
    re.IGNORECASE | re.MULTILINE,
)

# ── Nav skip ──────────────────────────────────────────────────────────────────
_NAV_SKIP_RE = re.compile(
    r'/(login|signin|logout|checkout|cart|account|search'
    r'|order|password|register|wishlist|compare)(/|$)',
    re.IGNORECASE,
)
# FIX 3: Bỏ thêm label vô nghĩa
_NAV_SKIP_LABELS = {"đọc tiếp", "xem thêm", "xem tất cả", "tài khoản", "gọi ngay", "chat"}


# ── Helpers ───────────────────────────────────────────────────────────────────

def _clean_md_text(text: str) -> str:
    """Bỏ markdown link syntax, giữ lại text thuần."""
    return re.sub(r'\[([^\]]+)\]\([^\)]+\)', r'\1', text).strip()


def _extract_contact(raw: str, base_url: str) -> dict[str, Any]:
    contact: dict[str, Any] = {}
    phones = _PHONE_RE.findall(raw)
    if phones:
        contact["phones"] = [p.strip() for p in phones]
    zalos = _ZALO_RE.findall(raw)
    if zalos:
        contact["zalo_ids"] = zalos
    emails = _EMAIL_RE.findall(raw)
    if emails:
        contact["emails"] = emails
    fb = _FB_RE.findall(raw)
    if fb:
        contact["facebook"] = [f"https://facebook.com/{h}" for h in fb]
    tiktok = _TIKTOK_RE.findall(raw)
    if tiktok:
        contact["tiktok"] = [f"https://tiktok.com/@{h}" for h in tiktok]
    ig = _INSTAGRAM_RE.findall(raw)
    if ig:
        contact["instagram"] = [f"https://instagram.com/{h}" for h in ig]
    addresses = _ADDRESS_RE.findall(raw)
    if addresses:
        contact["addresses"] = addresses[:3]
    hours = _HOURS_RE.findall(raw)
    if hours:
        contact["business_hours"] = list(set(hours))
    return contact


def _extract_nav_urls(raw: str, base_url: str) -> list[dict[str, str]]:
    domain = urlparse(base_url).netloc
    seen: set[str] = set()
    nav_urls: list[dict[str, str]] = []

    for label, href in _MD_LINK_RE.findall(raw):
        full = urljoin(base_url, href) if href.startswith("/") else href

        # Chỉ giữ internal link
        if urlparse(full).netloc != domain:
            continue
        # Bỏ UI path
        if _NAV_SKIP_RE.search(full):
            continue
        # FIX 3: Bỏ label vô nghĩa
        if label.strip().lower() in _NAV_SKIP_LABELS:
            continue
        # Bỏ duplicate
        if full in seen:
            continue

        seen.add(full)
        nav_urls.append({"url": full, "label": label.strip()})

    return nav_urls


def _extract_brand_identity(raw: str, url: str) -> dict[str, Any]:
    identity: dict[str, Any] = {}

    # FIX 1: Tách brand_name — lấy phần trước dấu " - " hoặc trước mô tả dài
    m = _H1_RE.search(raw)
    if m:
        h1_raw = _clean_md_text(m.group(1))
        # Tên thương hiệu = phần trước " - " đầu tiên nếu có, max 60 ký tự
        parts = re.split(r'\s[-–]\s', h1_raw, maxsplit=1)
        brand_name = parts[0].strip()
        # Nếu tên vẫn dài hơn 60 ký tự → lấy đến dấu phẩy hoặc chấm đầu tiên
        if len(brand_name) > 60:
            short = re.split(r'[,.]', brand_name)[0].strip()
            brand_name = short if len(short) > 3 else brand_name[:60]
        identity["brand_name"] = brand_name

        # Phần còn lại của H1 = description chính
        if len(parts) > 1:
            identity["description"] = parts[1].strip()

    # FIX 2: Tagline — chỉ lấy dòng text thuần, không chứa link
    taglines = _TAGLINE_SAFE_RE.findall(raw)
    clean_taglines = []
    for t in taglines:
        cleaned = _clean_md_text(t).strip()
        # Loại dòng còn sót markdown hoặc quá ngắn/dài
        if 20 < len(cleaned) < 300 and "](" not in cleaned:
            clean_taglines.append(cleaned)
    if clean_taglines:
        identity["taglines"] = clean_taglines[:3]

    return identity


def _extract_title(raw: str, url: str) -> str:
    m = _H1_RE.search(raw)
    if m:
        # FIX 1: Title = brand_name ngắn gọn
        h1 = _clean_md_text(m.group(1))
        parts = re.split(r'\s[-–]\s', h1, maxsplit=1)
        title = parts[0].strip()
        if len(title) > 3:
            return title[:200]
    m = re.search(r"https?://(?:www\.)?([^/]+)", url)
    return m.group(1) if m else url[:200]


def _compose_brand_text(
    raw: str,
    contact: dict[str, Any],
    identity: dict[str, Any],
    nav_urls: list[dict[str, str]],
    url: str,
) -> str:
    sections: list[str] = []

    if identity.get("brand_name"):
        sections.append(f"Tên thương hiệu: {identity['brand_name']}")
    if identity.get("description"):
        sections.append(f"Mô tả: {identity['description']}")
    if identity.get("taglines"):
        sections.append("Slogan: " + " | ".join(identity["taglines"]))

    if contact.get("phones"):
        sections.append("Điện thoại: " + ", ".join(contact["phones"]))
    if contact.get("zalo_ids"):
        sections.append("Zalo: " + ", ".join(contact["zalo_ids"]))
    if contact.get("emails"):
        sections.append("Email: " + ", ".join(contact["emails"]))
    if contact.get("addresses"):
        sections.append("Địa chỉ: " + " | ".join(contact["addresses"]))
    if contact.get("business_hours"):
        sections.append("Giờ làm việc: " + ", ".join(contact["business_hours"]))
    if contact.get("facebook"):
        sections.append("Facebook: " + ", ".join(contact["facebook"]))
    if contact.get("tiktok"):
        sections.append("TikTok: " + ", ".join(contact["tiktok"]))
    if contact.get("instagram"):
        sections.append("Instagram: " + ", ".join(contact["instagram"]))

    if nav_urls:
        # Tách collections và blogs riêng
        collections = [n for n in nav_urls if "/collections/" in n["url"]]
        blogs       = [n for n in nav_urls if "/blogs/" in n["url"] and "tin-tuc" not in n["url"]]
        posts       = [n for n in nav_urls if "/blogs/tin-tuc" in n["url"]]
        others      = [n for n in nav_urls if n not in collections + blogs + posts]

        if collections:
            sections.append("Danh mục sản phẩm:\n" + "\n".join(
                f"- {n['label']}: {n['url']}" for n in collections
            ))
        if blogs:
            sections.append("Chuyên mục blog:\n" + "\n".join(
                f"- {n['label']}: {n['url']}" for n in blogs
            ))
        if posts:
            sections.append("Bài viết nổi bật:\n" + "\n".join(
                f"- {n['label']}: {n['url']}" for n in posts[:5]
            ))
        if others:
            sections.append("Trang khác:\n" + "\n".join(
                f"- {n['label']}: {n['url']}" for n in others
            ))

    sections.append(f"Website: {url}")
    return "\n\n".join(sections)


print("✅ Cell 1 FIX loaded — chạy lại Cell 2 và Cell 3")

✅ Cell 1 FIX loaded — chạy lại Cell 2 và Cell 3


In [2]:
# ── CELL 2: Crawl + extract ───────────────────────────────────────────────────
import asyncio
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig
from crawl4ai.content_filter_strategy import PruningContentFilter
from crawl4ai.markdown_generation_strategy import DefaultMarkdownGenerator

TARGET_URL = "https://nongnghieppho.vn/"  # ← đổi URL ở đây

async def crawl_and_extract(url: str) -> dict:
    browser_cfg = BrowserConfig(headless=True, verbose=False)
    md_generator = DefaultMarkdownGenerator(
        content_filter=PruningContentFilter(threshold=0.48, threshold_type="fixed"),
    )
    run_cfg = CrawlerRunConfig(
        markdown_generator=md_generator,
        wait_until="networkidle",
        page_timeout=30_000,
        cache_mode="bypass",
    )

    async with AsyncWebCrawler(config=browser_cfg) as crawler:
        result = await crawler.arun(url=url, config=run_cfg)

    if not result.success:
        raise ValueError(f"Crawl thất bại: {result.error_message}")

    raw = (result.markdown.fit_markdown or result.markdown.raw_markdown or "").strip()
    if not raw:
        raise ValueError("Không có nội dung")

    # Pipeline extract
    contact  = _extract_contact(raw, url)
    identity = _extract_brand_identity(raw, url)
    nav_urls = _extract_nav_urls(raw, url)
    title    = _extract_title(raw, url)
    brand_text = _compose_brand_text(raw, contact, identity, nav_urls, url)

    return {
        "raw": raw,
        "title": title,
        "contact": contact,
        "identity": identity,
        "nav_urls": nav_urls,
        "brand_text": brand_text,
    }

result = await crawl_and_extract(TARGET_URL)
print("✅ Crawl xong")
print(f"   title     : {result['title']}")
print(f"   raw length: {len(result['raw'])} chars")
print(f"   nav_urls  : {len(result['nav_urls'])} links")
print(f"   contact   : {list(result['contact'].keys())}")

[FETCH]... ↓ https://nongnghieppho.vn/                                                                            | ✓ | ⏱: 6.36s 

[SCRAPE].. ◆ https://nongnghieppho.vn/                                                                            | ✓ | ⏱: 0.25s 

[COMPLETE] ● https://nongnghieppho.vn/                                                                            | ✓ | ⏱: 6.65s 

✅ Crawl xong
   title     : Nông Nghiệp Phố
   raw length: 4329 chars
   nav_urls  : 17 links
   contact   : ['phones', 'zalo_ids']


In [3]:
# ── CELL 3: Inspect kết quả ───────────────────────────────────────────────────
import json

print("=" * 60)
print("TITLE")
print("=" * 60)
print(result["title"])

print("\n" + "=" * 60)
print("IDENTITY")
print("=" * 60)
print(json.dumps(result["identity"], ensure_ascii=False, indent=2))

print("\n" + "=" * 60)
print("CONTACT")
print("=" * 60)
print(json.dumps(result["contact"], ensure_ascii=False, indent=2))

print("\n" + "=" * 60)
print(f"NAV URLS ({len(result['nav_urls'])} links)")
print("=" * 60)
for n in result["nav_urls"]:
    print(f"  [{n['label']}] → {n['url']}")

print("\n" + "=" * 60)
print("BRAND TEXT (sẽ ingest vào RAG)")
print("=" * 60)
print(result["brand_text"])

TITLE
Nông Nghiệp Phố

IDENTITY
{
  "brand_name": "Nông Nghiệp Phố",
  "description": "VTNN Nông Nghiệp Phố là chuỗi cửa hàng vật tư nông nghiệp mua sỉ giá tốt tại đô thị. Đầy đủ các loại: thuốc trừ sâu, phân bón hữu cơ, đất sạch, giá thể, chậu trồng rau"
}

CONTACT
{
  "phones": [
    "0865399086"
  ],
  "zalo_ids": [
    "1869973955811776365"
  ]
}

NAV URLS (17 links)
  [Khuyến mãi] → https://nongnghieppho.vn/collections/khuyen-mai
  [Đất sạch và Giá thể] → https://nongnghieppho.vn/collections/dat-sach-va-gia-the
  [Sản phẩm bán chạy] → https://nongnghieppho.vn/collections/san-pham-ban-chay
  [Phân Hữu Cơ] → https://nongnghieppho.vn/collections/phan-huu-co
  [Phòng trừ sâu bệnh] → https://nongnghieppho.vn/collections/phong-tru-sau-benh
  [Chăm sóc lan] → https://nongnghieppho.vn/collections/vat-tu-trong-lan
  [Chăm sóc hồng] → https://nongnghieppho.vn/collections/vat-tu-hoa-hong
  [Trồng rau sạch] → https://nongnghieppho.vn/collections/vat-tu-trong-rau
  [Chế phẩm vi sinh] → https:/

In [1]:
import sys
import os

# Trỏ thẳng về thư mục chứa gói 'app'
project_root = "/home/duoc/workflow/api"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Di chuyển thư mục làm việc của Jupyter về đây để tránh lỗi đường dẫn tương đối (nếu có)
os.chdir(project_root)

print("Cấu hình sys.path thành công! Thư mục hiện tại:", os.getcwd())


# 1. Khởi tạo các Service phụ thuộc (Giữ nguyên vì cần logic thật để cào web và embedding)
# ── VI. KÍCH HOẠT CHẠY THỬ NGHIỆM MOCK SERVICE NGAY TRONG JUPYTER ───────────────

import json  # <-- Thêm đầy đủ thư viện json ở đây
from unittest.mock import AsyncMock, MagicMock

# Import các module từ dự án
from app.rag.service import RAG
from app.rag.loader import DocumentLoader
from app.rag.business_service import BusinessCrawler

print("🧠 Đang khởi tạo các Dependency thật (RAG & Loader)...")
rag = RAG()
loader = DocumentLoader()

# Tạo Mock DB Session
db = MagicMock()
db.add = MagicMock()
db.commit = AsyncMock()
db.refresh = AsyncMock()

TARGET_URL = "https://nongnghieppho.vn"
MOCK_DOC_ID = 888

print(f"🌐 Bắt đầu kiểm thử Service...")
crawler = BusinessCrawler(rag=rag, loader=loader, db=db)

# Thực thi cào dữ liệu
await crawler.crawl_business(url=TARGET_URL, document_type="brand", document_id=MOCK_DOC_ID)

# Trích xuất "bắt" đối tượng DocumentPage truyền vào DB để xuất JSON kết quả
inserted_page = db.add.call_args[0][0]

print("\n" + "="*20 + " 🔍 KẾT QUẢ TRÍCH XUẤT TRONG EXTRACTED JSON " + "="*20)
print(json.dumps(inserted_page.extracted, indent=2, ensure_ascii=False))

Cấu hình sys.path thành công! Thư mục hiện tại: /home/duoc/workflow/api


/home/duoc/workflow/api/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
INFO:rag:[config] device=cpu dtype=torch.float32
INFO:rag:[tokenizer] fallback regex
INFO:business_crawler_v5:[BusinessCrawlerV5] Bắt đầu crawl: https://nongnghieppho.vn


🧠 Đang khởi tạo các Dependency thật (RAG & Loader)...
🌐 Bắt đầu kiểm thử Service...


[FETCH]... ↓ https://nongnghieppho.vn                                                                             | ✓ | ⏱: 4.63s 

[SCRAPE].. ◆ https://nongnghieppho.vn                                                                             | ✓ | ⏱: 0.25s 

...

[FETCH]... ↓ https://nongnghieppho.vn/ve-chung-toi                                                                | ✓ | ⏱: 3.58s 

[SCRAPE].. ◆ https://nongnghieppho.vn/ve-chung-toi                                                                | ✓ | ⏱: 0.14s 

[COMPLETE] ● https://nongnghieppho.vn/ve-chung-toi                                                                | ✓ | ⏱: 3.75s 

INFO:business_crawler_v5:[BusinessCrawlerV5] Crawl OK: https://nongnghieppho.vn/ve-chung-toi (826 chars)
INFO:rag:[load] 377 chunks restored
INFO:business_crawler_v5:[BusinessCrawlerV5] HOÀN THÀNH — page_id=None brand='Nông Nghiệp Phố' quality={'brand_core': 'very_high', 'brand_story': 'low', 'brand_visual': 'low', 'contact_cta': 'low', 'social_channels': 'low'}



==================== 🔍 KẾT QUẢ TRÍCH XUẤT TRONG EXTRACTED JSON ====================
{
  "source_url": "https://nongnghieppho.vn",
  "crawled_at": "2026-06-07T12:41:47.512833",
  "brand_identity": {
    "brand_name": "Nông Nghiệp Phố",
    "description": "VTNN Nông Nghiệp Phố là chuỗi cửa hàng vật tư nông nghiệp mua sỉ giá tốt tại đô thị. Đầy đủ các loại: thuốc trừ sâu, phân bón hữu cơ, đất sạch, giá thể, chậu trồng rau",
    "taglines": [
      "Th 7 23/05/2026",
      "Th 5 21/05/2026",
      "Th 5 14/05/2026"
    ],
    "mission": null,
    "vision": null,
    "story": null,
    "tone_keywords": [
      "chất lượng",
      "nhanh chóng"
    ],
    "main_products": [
      "Đất sạch và Giá thể",
      "Phân Hữu Cơ",
      "Phòng trừ sâu bệnh",
      "Chăm sóc lan",
      "Chăm sóc hồng",
      "Trồng rau sạch",
      "Chế phẩm vi sinh",
      "PHÂN BÓN"
    ],
    "certifications": [],
    "partners": []
  },
  "visual_identity": {
    "logos": [],
    "colors": []
  },
  "contact_de